In [1]:
# Importo las librerias antes que nada
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Cargo el dataset
df = pd.read_csv('./Data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [5]:
# Veo las primeras filas del dataset
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
# Veo las dimensiones del dataset y la información de las columnas
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.\n")
print("--- Información de Columnas y Tipos de Datos ---")
df.info()

Dimensiones del dataset: 7043 filas y 21 columnas.

--- Información de Columnas y Tipos de Datos ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   o

Verifico que este dataset esta compuesto por variables categoricas y algunas numericas. No contiene valores nulos.

In [8]:
# Verifico si hay valores duplicados
duplicated_rows = df.duplicated().sum()
if duplicated_rows > 0:
    print(f"Hay {duplicated_rows} filas duplicadas en el dataset.")

Tampoco hay valores duplicados.

In [13]:
# Detecto si hay textos en blanco o espacios vacíos
def buscar_espacios_vacios(df):
    espacios_por_columna = {}
    for col in df.select_dtypes(include=['object']).columns:
        # Contamos filas que contienen solo espacios en blanco
        cant_espacios = df[col].astype(str).str.strip().eq('').sum()
        if cant_espacios > 0:
            espacios_por_columna[col] = cant_espacios
            
    return pd.Series(espacios_por_columna, name="Cadenas Vacías / Espacios")

print("--- Columnas con espacios en blanco invisibles ---")
print(buscar_espacios_vacios(df))

--- Columnas con espacios en blanco invisibles ---
TotalCharges    11
Name: Cadenas Vacías / Espacios, dtype: int64


In [14]:
# Resumen de cada variable
resumen_columnas = []

for col in df.columns:
    resumen_columnas.append({
        'Columna': col,
        'Tipo_Dato': df[col].dtype,
        'Valores_Unicos': df[col].nunique(),
        'Ejemplos_Valores': list(df[col].dropna().unique()[:4]) 
    })

# Convierto la lista de diccionarios a un DataFrame para leerlo fácilmente
df_resumen = pd.DataFrame(resumen_columnas)
df_resumen

,Columna,Tipo_Dato,Valores_Unicos,Ejemplos_Valores
0,customerID,object,7043,"[7590-VHVEG, 5575-GNVDE, 3668-QPYBK, 7795-CFOCW]"
1,gender,object,2,"[Female, Male]"
2,SeniorCitizen,int64,2,"[0, 1]"
3,Partner,object,2,"[Yes, No]"
4,Dependents,object,2,"[No, Yes]"
5,tenure,int64,73,"[1, 34, 2, 45]"
6,PhoneService,object,2,"[No, Yes]"
7,MultipleLines,object,3,"[No phone service, No, Yes]"
8,InternetService,object,3,"[DSL, Fiber optic, No]"
9,OnlineSecurity,object,3,"[No, Yes, No internet service]"


In [15]:
# Reemplazo los espacios por NaN y los convierto a float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')

# Imputamos 0.0 a los clientes con tenure = 0
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# Mapeo SeniorCitizen a 'Yes' / 'No' para mantener consistencia
df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})

# Verifico que la columna ahora sea float
print("Nuevo tipo de dato de TotalCharges:", df['TotalCharges'].dtype)

Nuevo tipo de dato de TotalCharges: float64
